# 02 - Verify the public LiteLLM proxy

This notebook loads the generated connection details and checks that the public LiteLLM endpoint is healthy and can serve an authenticated Claude-compatible `/v1/messages` request through the configured Foundry model.

LiteLLM now uses PostgreSQL for bearer-key authentication, so this check validates the database-backed key path as well as the model gateway path.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

candidate_roots = [Path.cwd(), *Path.cwd().parents]
demo_root = next((root / 'demos' / 'litellm01-foundry-claude' for root in candidate_roots if (root / 'demos' / 'litellm01-foundry-claude').is_dir()), None)
if demo_root is None:
    raise RuntimeError('Could not locate demos/litellm01-foundry-claude from the notebook working directory.')
env_path = demo_root / 'env' / '.env'
load_dotenv(env_path)
required = ['LITELLM_PROXY_URL', 'LITELLM_MODEL_ALIAS']
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing {missing} in {env_path}. Run the deployment cell in 01_deploy_infra.ipynb successfully first.')
proxy_url = os.environ['LITELLM_PROXY_URL'].rstrip('/')
model_alias = os.environ['LITELLM_MODEL_ALIAS']
proxy_master_key = os.getenv('LITELLM_MASTER_KEY', '').strip()
proxy_key = (os.getenv('ANTHROPIC_AUTH_TOKEN', '').strip() or os.getenv('LITELLM_CLIENT_KEY', '').strip() or proxy_master_key)
if not proxy_key:
    raise RuntimeError(f'Missing API key in {env_path}. Expected ANTHROPIC_AUTH_TOKEN or LITELLM_CLIENT_KEY or LITELLM_MASTER_KEY.')
print(f'Proxy URL: {proxy_url}')
print(f'Model alias: {model_alias}')
print(f'Using key from env: {"ANTHROPIC_AUTH_TOKEN" if os.getenv("ANTHROPIC_AUTH_TOKEN") else ("LITELLM_CLIENT_KEY" if os.getenv("LITELLM_CLIENT_KEY") else "LITELLM_MASTER_KEY")}')

## Step 1 - Check health and authenticated Claude-compatible routing

This step first checks LiteLLM's public health endpoint, then sends a small authenticated request to /v1/messages using the generated database-backed client token. If the saved token is stale, it regenerates one from the master key and retries once.

A successful response confirms that PostgreSQL-backed key validation, LiteLLM routing, and the Foundry model deployment are all connected.

In [ ]:
import json
import urllib.error
import urllib.request

def http_get(url, headers=None, timeout=30):
    req = urllib.request.Request(url=url, headers=headers or {}, method='GET')
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.getcode(), resp.read().decode('utf-8')
    except urllib.error.HTTPError as exc:
        return exc.code, exc.read().decode('utf-8', errors='replace')

def http_post(url, payload, headers=None, timeout=180):
    req = urllib.request.Request(
        url=url,
        data=json.dumps(payload).encode('utf-8'),
        headers=headers or {'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.getcode(), resp.read().decode('utf-8')
    except urllib.error.HTTPError as exc:
        return exc.code, exc.read().decode('utf-8', errors='replace')

health_status, health_body = http_get(f'{proxy_url}/health/liveliness', timeout=30)
if not (200 <= health_status < 300):
    raise RuntimeError(f'Health check failed ({health_status}): {health_body[:500]}')

smoke_payload = {
    'model': model_alias,
    'max_tokens': 32,
    'messages': [{'role': 'user', 'content': 'Reply with the single word OK.'}],
}

def call_messages(api_key):
    return http_post(
        f'{proxy_url}/v1/messages',
        smoke_payload,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        timeout=180,
    )

smoke_status, smoke_body = call_messages(proxy_key)

# If the saved client token is stale (or only master key exists), regenerate a DB-backed token and retry once.
if smoke_status == 401 and 'token_not_found_in_db' in smoke_body and proxy_master_key:
    keygen_status, keygen_body = http_post(
        f'{proxy_url}/key/generate',
        {'duration': '7d', 'models': [model_alias], 'max_budget': 50, 'spend': 50},
        headers={'Authorization': f'Bearer {proxy_master_key}', 'Content-Type': 'application/json'},
        timeout=60,
    )
    if 200 <= keygen_status < 300:
        refreshed_key = (json.loads(keygen_body) or {}).get('key', '').strip()
        if refreshed_key:
            proxy_key = refreshed_key
            smoke_status, smoke_body = call_messages(proxy_key)

            # Persist refreshed token for next runs.
            lines = []
            if env_path.exists():
                lines = env_path.read_text(encoding='utf-8').splitlines()
            replaced_auth = False
            replaced_client = False
            for idx, line in enumerate(lines):
                if line.startswith('ANTHROPIC_AUTH_TOKEN='):
                    lines[idx] = f'ANTHROPIC_AUTH_TOKEN={proxy_key}'
                    replaced_auth = True
                elif line.startswith('LITELLM_CLIENT_KEY='):
                    lines[idx] = f'LITELLM_CLIENT_KEY={proxy_key}'
                    replaced_client = True
            if not replaced_client:
                lines.append(f'LITELLM_CLIENT_KEY={proxy_key}')
            if not replaced_auth:
                lines.append(f'ANTHROPIC_AUTH_TOKEN={proxy_key}')
            env_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
            print('Refreshed DB-backed API token and updated env/.env.')

if not (200 <= smoke_status < 300):
    raise RuntimeError(
        f'Claude-compatible smoke test failed ({smoke_status}): {smoke_body[:500]}'
    )

print('Health: OK')
print(f'Claude-compatible model request: {model_alias} OK')
print('Note: /v1/models discovery is not checked because LiteLLM requires a database for that route in this configuration.')